In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image
import time
import threading
import requests

In [ ]:
model_id = "llava-hf/llava-1.5-7b-hf"
image_url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg"

In [ ]:
timestamps = []
vram_usage = []
start_time = None
monitoring = False
markers = []

In [ ]:
def monitor_vram(interval=0.05):
    """Background thread to monitor vRAM"""
    global monitoring, timestamps, vram_usage, start_time
    while monitoring:
        if torch.cuda.is_available():
            current_time = (time.time() - start_time) * 1000
            vram_mb = torch.cuda.memory_allocated() / (1024 ** 2)
            timestamps.append(current_time)
            vram_usage.append(vram_mb)
        time.sleep(interval)

def add_marker(label):
    """Add a marker at current time"""
    if torch.cuda.is_available():
        current_time = (time.time() - start_time) * 1000
        current_vram = torch.cuda.memory_allocated() / (1024 ** 2)
        markers.append((current_time, current_vram, label))
        print(f"[{current_time:.1f}ms] {label}: {current_vram:.1f} MB")

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA not available. GPU required for profiling.")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

monitoring = True
start_time = time.time()
monitor_thread = threading.Thread(target=monitor_vram, daemon=True)
monitor_thread.start()

time.sleep(0.1)
add_marker("Start")

- **Loading Processor**

In [ ]:
processor = AutoProcessor.from_pretrained(model_id)
add_marker("Processor Loaded")

- **Loading Model**

In [ ]:
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
add_marker("Model Loaded")

time.sleep(0.2)

- **Loading image**

In [ ]:
image = Image.open(requests.get(image_url, stream=True).raw)
add_marker("Image Loaded")

- **Preparing embeds for text and image**

In [ ]:
prompt = "USER: <image>\nWhat is in this image?\nASSISTANT:"
inputs = processor(text=prompt, images=image, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}
add_marker("Inputs Prepared")

- **Model Inference**

In [ ]:
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=50)
add_marker("Inference Complete")

- **Decode Step**

In [ ]:
generated_text = processor.decode(outputs[0], skip_special_tokens=True)
print(f"\nGenerated: {generated_text[:100]}...")

time.sleep(0.2)

In [ ]:
model_loaded_val = next((v for t, v, l in markers if l == "Model Loaded"), 0)
max_val = max(vram_usage)

y_zoom_min = model_loaded_val - 200
y_zoom_max = max_val + 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={'height_ratios': [1, 2]})

ax1.plot(timestamps, vram_usage, 'b-', linewidth=1.5)
ax1.set_title(f'Overview: {model_id.split("/")[-1]}', fontsize=12, fontweight='bold')
ax1.set_ylabel('vRAM (MB)')
ax1.grid(True, alpha=0.3)

ax2.plot(timestamps, vram_usage, 'b-', linewidth=2)
ax2.fill_between(timestamps, vram_usage, alpha=0.1, color='blue')

ax2.set_ylim(y_zoom_min, y_zoom_max)
ax2.set_title('Detail View: Inference & Processing Shifts', fontsize=12, fontweight='bold')
ax2.set_xlabel('Time (ms)', fontsize=12)
ax2.set_ylabel('vRAM (MB) - Zoomed', fontsize=12)
ax2.grid(True, alpha=0.4, which='both')

colors = ['red', 'green', 'orange', 'purple', 'brown', 'cyan']
for ax in [ax1, ax2]:
    for i, (t, vram, label) in enumerate(markers):
        color = colors[i % len(colors)]
        ax.axvline(x=t, color=color, linestyle='--', alpha=0.7)
        if ax == ax1 or (y_zoom_min <= vram <= y_zoom_max):
            ax.plot(t, vram, 'o', color=color, markersize=8, label=label if ax == ax1 else "")

ax1.legend(loc='lower right', fontsize=9)

inference_delta = max_val - model_loaded_val
stats_text = (f"Base Model: {model_loaded_val:.1f} MB\n"
              f"Peak Usage: {max_val:.1f} MB\n"
              f"Inference Overhead: +{inference_delta:.1f} MB")

ax2.text(0.02, 0.95, stats_text, transform=ax2.transAxes,
        verticalalignment='top', bbox=dict(boxstyle='round',
        facecolor='wheat', alpha=0.5), fontsize=10)

plt.tight_layout()

plt.savefig('../metrics/vram_profile_inference.png')

plt.show()